In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, min as spark_min, max as spark_max, avg, expr,
    first, last, date_trunc, date_add, weekofyear, year
)
from pyspark.sql.functions import current_date, date_format, current_timestamp

SCHEMA_FROM = "SILVER"
TB_NAME_FROM = "TB_BITCOIN"
TB_FROM = f"{SCHEMA_FROM}.{TB_NAME_FROM}"

SCHEMA_DESTINY = "GOLD"
TB_NAME = "TB_BITCOIN_WEEKLY"
TB_DESTINY = f"{SCHEMA_DESTINY}.{TB_NAME}"

In [0]:
df_bitcoin = spark.read.table(TB_FROM)

df_bitcoin_filtered_last_year = df_bitcoin.filter(
    col("DT_DATA") >= expr("current_date() - INTERVAL 1 YEAR")
)

#indetify week
df_bitcoin_weekly = (df_bitcoin_filtered_last_year
    .withColumn("DT_WEEK_START", date_trunc("week", col("DT_DATA")).cast("date"))
    .withColumn("DT_WEEK_END", date_add(col("DT_WEEK_START"), 6))
    .withColumn("NR_WEEK", weekofyear(col("DT_DATA"))))

#groupby weekly  
df_weekly = df_bitcoin_weekly.groupBy("DT_WEEK_START", "DT_WEEK_END").agg(
    first("DT_DATA").alias("DT_DATA"),
    first("VL_OPEN").alias("VL_OPEN"),
    first("VL_CLOSE").alias("VL_CLOSE"),
    spark_max("VL_HIGH").alias("VL_HIGH_MAX"),
    spark_min("VL_LOW").alias("VL_LOW_MAX"),
    expr("percentile_approx(VL_VOLUME, 0.5)").alias("VL_VOLUME_MEDIAN")
)

#calculate percentage variation
df_variation = df_weekly.withColumn("VL_VARIATION_PCT", (col("VL_CLOSE") - col("VL_OPEN")) / col("VL_OPEN") * 100)

#write table
df_variation.write.format('delta').mode('overwrite').saveAsTable(TB_DESTINY)